# 1. AIND Ephys Job Dispatch

Build an `AINDEPhysDispatchScanConfig`, expand it with `GridScanGenerationTask`, and run `AINDEPhysDispatchTask` for each coordinate. The task reads every raw ElectricalSeries in the NWB file's acquisition and writes one `job_<i>.json` per series, segment and probe group into the coordinate's output folder (`obi-output/01_aind_ephys_dispatch/grid_scan/0/`). Each job holds its recording serialized relative to the NWB file's folder, recorded as `input_folder`.

Reads the toy NWB file written by notebook 00.

## Imports

In [1]:
import json
from pathlib import Path

from obi_one.core.run_tasks import run_tasks_for_generated_scan
from obi_one.core.scan_generation import GridScanGenerationTask
from obi_one.scientific.tasks.aind_ephys._01_dispatch.config import AINDEPhysDispatchScanConfig

## Build the scan config

In [2]:
obi_output = (Path.cwd() / "../../../../../../obi-output").resolve()
nwb_path = obi_output / "00_toy_example_recording" / "toy_recording.nwb"
assert nwb_path.exists(), f"{nwb_path} not found. Run 00_generate_toy_recording.ipynb first."

scan_config = AINDEPhysDispatchScanConfig(
    initialize=AINDEPhysDispatchScanConfig.Initialize(nwb_path=nwb_path),
)

## Generate the grid scan and run the dispatch task

In [3]:
grid_scan = GridScanGenerationTask(
    form=scan_config,
    output_root=obi_output / "01_aind_ephys_dispatch" / "grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

run_tasks_for_generated_scan(grid_scan)

/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-one/.venv/lib/python3.12/site-packages/spikeinterface/core/baserecordingsnippets.py:110: FutureWarning: The 'in_place' argument is deprecated and will be removed in version 0.106.0. The `set_probe/probegroup()` are now always in place; please remove the in_place argument.
  return self.set_probegroup(probegroup, group_mode=group_mode, in_place=in_place)


[PosixPath('/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-output/01_aind_ephys_dispatch/grid_scan/0')]

## Inspect the jobs

In [4]:
coord_dir = Path(grid_scan.single_configs[0].coordinate_output_root)
for job_file in sorted(coord_dir.glob("job_*.json")):
    job = json.loads(job_file.read_text())
    print(job_file.name, "->", job["recording_name"], f"({job['duration']} s)")
    print("  input_folder:", job["input_folder"])

job_0.json -> block0_acquisition-ElectricalSeriesRaw_recording1 (10.0 s)
  input_folder: /Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-output/00_toy_example_recording
